# Notebook 01 — Data Exploration

Explore the running form dataset: class balance, keypoint quality, clip lengths, pose visibility.

**Steps:**
1. Load labels and check class distribution
2. Inspect pose CSV quality (detection rates, visibility)
3. Analyse clip duration and frame counts
4. Visualise sample keypoints
5. Summary statistics

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

CLASS_PALETTE = {
    'good_form':    '#2ecc71',
    'overstriding': '#e74c3c',
    'forward_lean': '#f39c12',
    'arm_crossing': '#9b59b6',
}
CLASS_ORDER = ['good_form', 'overstriding', 'forward_lean', 'arm_crossing']
print('✅ Libraries loaded')

## 1. Load Labels & Class Distribution

In [ ]:
labels_path = Path('../data/annotations/form_labels.csv')
labels = pd.read_csv(labels_path) if labels_path.exists() else pd.DataFrame()

if not labels.empty:
    print(f'Total clips: {len(labels)}')
    print('Class distribution:')
    dist = labels['form_class'].value_counts()
    print(dist)
    print(f'\nLabel sources: {labels["label_source"].value_counts().to_dict()}')
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # Bar chart
    counts = dist.reindex(CLASS_ORDER).fillna(0)
    bars = axes[0].bar(counts.index, counts.values,
                       color=[CLASS_PALETTE[c] for c in counts.index],
                       edgecolor='white', width=0.55)
    for bar, val in zip(bars, counts.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     str(int(val)), ha='center', fontweight='bold')
    axes[0].set_title('Clips per Form Class'); axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=15)

    # Pie chart
    axes[1].pie(counts.values,
                labels=[c.replace('_', '\n') for c in counts.index],
                colors=[CLASS_PALETTE[c] for c in counts.index],
                autopct='%1.1f%%', startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    axes[1].set_title('Proportions')
    
    plt.suptitle('Dataset Class Distribution', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No labels found. Run label_clips.py first.')

## 2. Pose Detection Quality

In [ ]:
manifest_path = Path('../data/raw/poses/poses_manifest.json')
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    
    df_manifest = pd.DataFrame(manifest)
    print(f'Videos processed: {len(df_manifest)}')
    print(f'Mean detection rate: {df_manifest["detection_rate"].mean():.1%}')
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    # Detection rate histogram
    axes[0].hist(df_manifest['detection_rate'], bins=20,
                 color='#4C72B0', edgecolor='white', alpha=0.85)
    axes[0].axvline(0.9, color='green', linestyle='--', label='0.9 threshold')
    axes[0].set_title('Pose Detection Rate Distribution')
    axes[0].set_xlabel('Detection Rate')
    axes[0].legend()
    
    # Detection rate by form class
    if 'form_class' in df_manifest.columns:
        for cls in CLASS_ORDER:
            sub = df_manifest[df_manifest['form_class'] == cls]['detection_rate']
            if len(sub) > 0:
                axes[1].bar(cls, sub.mean(), color=CLASS_PALETTE[cls],
                           edgecolor='white', alpha=0.85)
                axes[1].errorbar(cls, sub.mean(), yerr=sub.std(),
                                fmt='none', color='black', capsize=4)
        axes[1].set_title('Mean Detection Rate by Class')
        axes[1].set_ylabel('Detection Rate')
        axes[1].set_ylim(0, 1.1)
        axes[1].tick_params(axis='x', rotation=15)
    
    plt.suptitle('Pose Extraction Quality', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Low quality clips
    low_qual = df_manifest[df_manifest['detection_rate'] < 0.7]
    if len(low_qual) > 0:
        print(f'\n⚠️  {len(low_qual)} clips with < 70% detection rate:')
        print(low_qual[['video', 'form_class', 'detection_rate']].to_string())
    else:
        print('\n✅ All clips have good detection rates (≥ 70%)')
else:
    print('Run extract_poses.py first.')

## 3. Clip Duration & Frame Counts

In [ ]:
if 'df_manifest' in dir() and not df_manifest.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    # Frame counts
    axes[0].hist(df_manifest['total_frames'], bins=25,
                 color='#55A868', edgecolor='white', alpha=0.85)
    axes[0].axvline(30, color='orange', linestyle='--', label='30 frames (seq_len)')
    axes[0].set_title('Clip Length Distribution')
    axes[0].set_xlabel('Total Frames')
    axes[0].legend()
    
    # Duration (seconds)
    df_manifest['duration_s'] = df_manifest['total_frames'] / df_manifest['fps'].fillna(30)
    axes[1].hist(df_manifest['duration_s'], bins=25,
                 color='#8172B2', edgecolor='white', alpha=0.85)
    axes[1].axvline(1.0, color='orange', linestyle='--', label='1s minimum')
    axes[1].set_title('Clip Duration Distribution')
    axes[1].set_xlabel('Duration (seconds)')
    axes[1].legend()
    
    plt.suptitle('Clip Metadata', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Median duration : {df_manifest['duration_s'].median():.1f}s")
    print(f"Short clips (<1s): {(df_manifest['duration_s'] < 1).sum()}")
    print(f"Long clips (>10s): {(df_manifest['duration_s'] > 10).sum()}")

## 4. Keypoint Visibility Analysis

In [ ]:
import glob

# Load one sample pose CSV to check visibility quality
pose_csvs = glob.glob('../data/raw/poses/**/*_poses.csv', recursive=True)

if pose_csvs:
    sample = pd.read_csv(pose_csvs[0]).head(100)
    vis_cols = [c for c in sample.columns if c.endswith('_vis')]
    
    # Running-relevant landmarks only
    running_lms = ['nose', 'left_shoulder', 'right_shoulder',
                   'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist',
                   'left_hip', 'right_hip', 'left_knee', 'right_knee',
                   'left_ankle', 'right_ankle', 'left_heel', 'right_heel']
    running_vis = [f'{lm}_vis' for lm in running_lms if f'{lm}_vis' in sample.columns]
    
    mean_vis = sample[running_vis].mean().sort_values(ascending=True)
    colors = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.8 else '#2ecc71'
              for v in mean_vis.values]
    
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh([c.replace('_vis','') for c in mean_vis.index],
            mean_vis.values, color=colors, edgecolor='white')
    ax.axvline(0.5, color='red', linestyle='--', alpha=0.7, label='0.5 threshold')
    ax.axvline(0.8, color='orange', linestyle='--', alpha=0.7, label='0.8 threshold')
    ax.set_title('Mean Keypoint Visibility (running landmarks)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Mean Visibility [0-1]'); ax.set_xlim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    low_vis = [c for c, v in zip(mean_vis.index, mean_vis.values) if v < 0.5]
    print(f'Low visibility landmarks (< 0.5): {[c.replace("_vis","") for c in low_vis]}' if low_vis else '✅ All landmarks OK')
else:
    print('Run extract_poses.py first to generate pose CSVs.')

## 5. Summary Statistics

In [ ]:
feat_path = Path('../data/processed/features/biomech_features.csv')
if feat_path.exists():
    df = pd.read_csv(feat_path)
    print(f'Total rows      : {len(df):,}')
    print(f'Total clips     : {df["video_stem"].nunique()}')
    print(f'Feature columns : {len(df.columns)}')
    print('\nClips per class:')
    print(df.groupby('form_class')['video_stem'].nunique())
    print('\nMean trunk lean by class:')
    print(df.groupby('form_class')['trunk_lean_angle'].mean().round(2))
else:
    print('Run biomech_features.py first.')